In [1]:
import sys
print(sys.executable)

/Users/<user>/code/jolennd/bootcamp/neueFische/projects/michel-capstone-housing-risk-main/.venv/bin/python


In [2]:
import os
import pandas as pd
import psycopg2
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
conn = psycopg2.connect()  # libpq reads PGHOST/PGPORT/PGDATABASE/PGUSER/PGPASSWORD automatically
engine = create_engine(
    f"postgresql://{os.getenv('PGUSER')}:{os.getenv('PGPASSWORD')}@{os.getenv('PGHOST')}:{os.getenv('PGPORT')}/{os.getenv('PGDATABASE')}"
)

In [3]:
zhvi = pd.read_sql("SELECT county_fips, month, zhvi FROM zhvi_monthly", engine)
mortgage = pd.read_sql("SELECT month, mortgage_rate FROM mortgage_monthly", engine)
print(f"zhvi_monthly: {len(zhvi):,} rows, {zhvi['county_fips'].nunique():,} counties, {zhvi['month'].nunique():,} months")
print(f"  range: {zhvi['month'].min()} to {zhvi['month'].max()}")
print(f"mortgage_monthly: {len(mortgage):,} rows, range {mortgage['month'].min()} to {mortgage['month'].max()}")

zhvi_monthly: 709,996 rows, 3,071 counties, 320 months
  range: 2000-01-31 to 2026-08-31
mortgage_monthly: 665 rows, range 1971-04-30 to 2026-08-31


Reference only, this is the contents of `sql/002_county_month.sql`, run by the next cell via `engine.exec_driver_sql(...)`. Not meant to be executed directly as a Python cell.

```sql
-- Zillow is the spine (LEFT JOIN): a month missing from mortgage_monthly should
-- surface as a NULL mortgage_rate, never as a silently dropped county-month row.
CREATE TABLE IF NOT EXISTS county_month_check AS
SELECT z.county_fips, z.month, z.zhvi, m.mortgage_rate
FROM zhvi_monthly AS z
LEFT JOIN mortgage_monthly AS m ON m.month = z.month;
```

In [4]:
with engine.begin() as c:
    c.exec_driver_sql(open("sql/002_county_month.sql").read())

check = pd.read_sql("SELECT * FROM county_month_check", engine)
missing_counties = set(zhvi["county_fips"]) - set(check["county_fips"])
null_rates = check["mortgage_rate"].isna().sum()

print(f"county_month_check: {len(check):,} rows")
assert missing_counties == set(), f"STOP: counties dropped in the join: {missing_counties}"
assert len(check) == len(zhvi), "STOP: row count changed across the join"
print(f"NULL mortgage_rate rows: {null_rates:,} (expected only for months missing from FRED, if any)")

county_month_check: 709,996 rows
NULL mortgage_rate rows: 0 (expected only for months missing from FRED, if any)


In [5]:
reference = pd.read_sql("SELECT * FROM county_month ORDER BY county_fips, month", engine)
yours = check.sort_values(["county_fips", "month"]).reset_index(drop=True)
pd.testing.assert_frame_equal(reference.reset_index(drop=True), yours, check_like=True)
print("Matches the reference county_month table exactly.")

Matches the reference county_month table exactly.
